# 🚗 Training & Benchmark Pipeline: Deteksi Pedestrian Low-Light (YOLOv8)

> **Mata Kuliah:** Visi Komputer (IF0PP006) — Semester Ganjil 2026/2027  
> **Kelompok:** 9 (Krisantus Linardi & Kristofer Leonard)  
> **Platform:** Google Colab (GPU NVIDIA T4)

---

## 📌 Ringkasan Skenario Eksperimen

| Eksperimen | Model | Metode & Konfigurasi | Durasi Colab T4 |
|:---:|---|---|:---:|
| **Exp-D (Baru)** | YOLOv8n Robust | **Freeze Backbone (10), lr0=0.001, Multi-Scale** | **~15–20 Menit** (Rekomendasi Utama) |
| **Exp-A** | YOLOv8n Standard | Fine-Tuning Default Standar | *Sudah Selesai Dilatih* |
| **Exp-B** | YOLOv8n CLAHE | Prapemrosesan L-channel CLAHE | *Sudah Selesai Dilatih* |
| **Exp-C** | YOLOv8n Gamma | Prapemrosesan Gamma Correction ($\gamma=0.6$) | *Sudah Selesai Dilatih* |
| **B0** | YOLOv8n COCO | Zero-Shot Baseline (Tanpa Training) | *Instan* |


## 1. Verifikasi Akselerasi GPU & Instalasi Dependensi
*(Wajib dijalankan - Membutuhkan ~1 menit)*

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas tqdm


## 2. Setup Dataset Low-Light (Roboflow)
*(Wajib dijalankan - Membutuhkan ~1 menit)*
Unggah `Low-light.v2i.yolov8.zip` ke Google Colab, lalu jalankan sel ini untuk mengekstrak.

In [ ]:
import os
import zipfile
from pathlib import Path

zip_path = Path("Low-light.v2i.yolov8.zip")
dataset_dir = Path("dataset")

if zip_path.exists() and not dataset_dir.exists():
    print("Mengekstrak dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("Ekstraksi selesai!")
else:
    print("Direktori dataset sudah siap:", dataset_dir)


## 3. [JALUR CEPAT & UTAMA] Pelatihan Eksperimen D (Robust Transfer Learning)
*(Wajib dijalankan jika ingin model baru anti-catastrophic forgetting — Hanya ~15–20 menit)*

**Keunggulan Konfigurasi Exp-D:**
- `freeze=10`: Mengunci 10 layer pertama backbone COCO agar tidak lupa siluet manusia umum.
- `lr0=0.001`: Learning rate halus agar tidak merusak bobot pretrained.
- `scale=0.5`: Multi-scale jitter agar mendeteksi orang dekat (besar) maupun jauh (kecil).
- `mosaic=1.0`: Mengatasi oklusi pejalan kaki di jalanan malam.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
IMGSZ = 640
BATCH = 16
SEED = 42

print("\n🚀 Memulai Pelatihan Exp-D (Robust Fine-Tuning)...")
model_d = YOLO("yolov8n.pt")
model_d.train(
    data="dataset/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    freeze=10,             # KUNCI 10 layer pertama backbone COCO
    lr0=0.001,             # Learning rate 10x lebih halus
    lrf=0.01,              # Cosine decay factor
    scale=0.5,             # Multi-scale jitter (objek kecil s/d besar)
    mosaic=1.0,            # Mosaic augmentation
    project="runs_comvis",
    name="Exp_D_Robust",
    save=True,
    device=0
)

print("\n✅ Pelatihan Exp-D Selesai! Bobot tersimpan di: runs_comvis/Exp_D_Robust/weights/best.pt")


## 4. Evaluasi Kinerja Model Exp-D
*(Jalankan setelah Sel 3 selesai untuk melihat metrik Recall dan mAP)*

In [ ]:
# Evaluasi pada split validasi
val_d = model_d.val(data="dataset/data.yaml", split="val", device=0)

print("\n=== HASIL EVALUASI MODEL EXP-D ===")
print(f"Recall Pejalan Kaki : {val_d.results_dict.get('metrics/recall(B)', 0.0):.4f}")
print(f"mAP@50              : {val_d.results_dict.get('metrics/mAP50(B)', 0.0):.4f}")
print(f"mAP@50-95           : {val_d.results_dict.get('metrics/mAP50-95(B)', 0.0):.4f}")


## 5. [OPSIONAL / HISTORIS] Pelatihan Model A, B, dan C (~1 Jam)
> ⚠️ **CATATAN PENTING:** Sel ini **TIDAK PERLU DIJALANKAN** jika Anda hanya ingin model baru Exp-D, karena bobot `Exp_A_Standard_best.pt`, `Exp_B_CLAHE_best.pt`, dan `Exp_C_Gamma_best.pt` sudah selesai dilatih dan tersimpan di repositori!

In [ ]:
# LEWATI SEL INI JIKA SUDAH MEMILIKI BOBOT EXP-A, B, DAN C
# Hapus tanda kutip triple jika memang ingin melatih ulang semuanya (~1 jam):
"""
# Exp-A
model_a = YOLO('yolov8n.pt')
model_a.train(data='dataset/data.yaml', epochs=50, imgsz=640, batch=16, seed=42, project='runs_comvis', name='Exp_A_Standard', device=0)
"""
